In [1]:
import os
if 'PATH_SET' not in locals():
    os.chdir('..')
    PATH_SET = True
    
import sys
import numpy as np

import torch
import zuko
    
from geobed.utils.sample_distribution import SampleDistribution
from geobed import BED_base_explicit, BED_base_nuisance


from helpers.geographic_setup import (
    design_space_full, design_space_shoulder,
    events_shoulder, events_full
)    


from helpers.likelihood import (
    DataLikelihoodAttenuation,
    DataLikelihood,
    logistic_picking_likelihood_tt)

from helpers.forward import TTLookup

if 'THREADS_SET' not in locals():
    try:
        os.environ['OMP_NUM_THREADS'] = '1'
        os.environ['MKL_NUM_THREADS'] = '1'
        
        torch.set_num_threads(1)
        torch.set_num_interop_threads(1)
        THREADS_SET = True
    except:
        pass

from tqdm.auto import tqdm

import matplotlib.pyplot as plt
import matplotlib.style as style
style.use('https://raw.githubusercontent.com/dominik-strutz/dotfiles/main/mystyle.mplstyle')

In [2]:
design_space = design_space_full

model_prior_samples = torch.load('data/priors/prior_samples_full_disp.pt')
model_prior_sample_dist = SampleDistribution(
    model_prior_samples)

forward_function = TTLookup(
    model_prior_samples, design_space_full,
    torch.load('data/data_lookup/gradient_full_disp.pt'),)


data_likelihood = DataLikelihood(
    forward_function=forward_function,
    dependence_distance=100.0,
    vel_sigma=0.05,
    tt_obs_std=0.01, 
)

BED_class = BED_base_explicit(
    data_likelihood_func=data_likelihood,
    m_prior_dist=model_prior_sample_dist,
)

In [3]:
N_designs = 1
N_rec = 3

torch.manual_seed(0)
designs = torch.stack([
    design_space[
        torch.randperm(len(design_space))[:N_rec]] 
    for _ in range(N_designs)])

In [4]:
for _ in tqdm(range(10)):
    eig, info = BED_class.calculate_EIG(
            designs,
            eig_method='NMC',
            eig_method_kwargs=dict(
                N = 500,
                reuse_M = True,
                # memory_efficient = True,
                memory_efficient = False,            
                ),
            random_seed=0,
            progress_bar=True,
            )
    print(eig)

  0%|          | 0/10 [00:00<?, ?it/s]

tensor(1.6545)
tensor(1.6545)
tensor(1.6545)
tensor(1.6545)
tensor(1.6545)
tensor(1.6545)
tensor(1.6545)
tensor(1.6545)
tensor(1.6545)
tensor(1.6545)


In [5]:
for _ in tqdm(range(10)):
    eig, info = BED_class.calculate_EIG(
            designs,
            eig_method='NMC',
            eig_method_kwargs=dict(
                N = 500,
                reuse_M = True,
                memory_efficient = True,
                # memory_efficient = False,            
                ),
            random_seed=0,
            progress_bar=True,
            )
    print(eig)

  0%|          | 0/10 [00:00<?, ?it/s]

tensor(1.6545)
tensor(1.6545)
tensor(1.6545)
tensor(1.6545)
tensor(1.6545)
tensor(1.6545)
tensor(1.6545)
tensor(1.6545)
tensor(1.6545)
tensor(1.6545)
